# Grounding a Translator answer in public datasets — asthma

**Question.** NCATS Translator answers *"what might treat asthma?"* with `drug → gene → disease`
paths. Each `drug → gene` hop is a mechanistic claim. Can we find, in the NIAID Data Ecosystem
(NDE) and the repositories it indexes, primary data that speaks to those claims?

Four strategies, ordered here from the most direct test to the most permissive:

| | strategy | question | source |
|---|---|---|---|
| **1** | activity | does **activity** data support the claim the edge actually makes? | PubChem BioAssay + ChEMBL |
| **2** | expression, precomputed | does the drug change the gene's **expression**? | GXA contrasts, via NDE `@type:Inference` |
| **3** | expression, inverted | what **other** compounds move this gene? | the same GXA contrasts, queried gene-first |
| **4** | dataset discovery | is there a **dataset** measuring expression after this drug? | GEO, discovered through NDE `@type:Sample` |

Strategy 1 asks the question Translator's edges actually pose. Strategy 2 asks the question the
project originally set out to ask, and largely cannot answer. Strategy 3 turns that failure into a
different question. Strategy 4 gives up on adjudicating any single edge and just looks for relevant
data.

Asthma is used throughout. Everything reads from archived artifacts under `data/ars/<pk>/`, so the
numbers are reproducible; ARS answers change over time and the archived `paths.json` is the citable
object.

> **Naming.** These are called *Routes D, A, C, B* respectively in
> [`results/REPORT.md`](../results/REPORT.md) and in the `route_*.json` artifacts, which were
> lettered in the order they were built rather than the order that reads best. The mapping is
> Strategy 1 = Strategy 1, 2 = Strategy 2, 3 = Strategy 3, 4 = Strategy 4.

In [1]:
import json, collections
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 60)
PK = "5b656c0f-b7da-4db4-ba1f-d3a794b422d4"      # ARS query id for asthma, 2026-09-01
ARS = Path("../data/ars") / PK

paths = json.loads((ARS / "paths.json").read_text())["paths"]
print(f"asthma (MONDO:0004979) — ARS pk {PK}")
print(f"{len(paths):,} drug→gene→disease paths")

asthma (MONDO:0004979) — ARS pk 5b656c0f-b7da-4db4-ba1f-d3a794b422d4
796 drug→gene→disease paths


## Background — the NIAID Data Ecosystem

The [NIAID Data Ecosystem](https://data.niaid.nih.gov) (NDE) is a **discovery layer over other
people's repositories**. It does not host data. It harvests metadata from source archives on a
schedule, normalises the result to a common schema.org-based model with ontology-mapped fields
(MONDO for disease, NCBI Taxon for species, EDAM for measurement technique), and serves it through
a BioThings/Elasticsearch API. The practical consequence for this notebook is that one query
reaches 48 repositories at once, and the disease axis is already ontology-mapped — the MONDO
mapping was itself done using Translator Knowledge Provider APIs, so the two systems already share
a vocabulary on that one axis.

**The scope is NIAID's, the technology is not.** What gets indexed is chosen for NIAID's mission —
infectious, immunologic and allergic disease — and that shapes which repositories are harvested and
which records are prioritised. But nothing in the harvest → normalise → index stack is specific to
that biology; it is a generic metadata pipeline, and the same machinery would serve any domain with
a different source list. So the coverage limits found below are curation decisions, not
architectural ones.

In [2]:
import sys
sys.path.insert(0, "../src")
from translator_nde.nde import NDEClient, PROD, STAGING

nde = NDEClient(base_url=PROD)
nde_staging = NDEClient(base_url=STAGING)

print(f"production index: {nde.count('__all__'):,} records across "
      f"{len(nde.facet('@type:Dataset', 'includedInDataCatalog.name', facet_size=1000))} repositories\n")

counts = pd.DataFrame([
    {"@type": t, "production": nde.count(f"@type:{t}"), "what it is": w}
    for t, w in [("Dataset", "a study / series / accession"),
                 ("Sample", "one biological sample, e.g. a GEO GSM"),
                 ("ComputationalTool", "software and workflows"),
                 ("ResourceCatalog", "a whole repository, described as a record")]
])
# Inference and DataCollection exist only on staging; querying prod returns 0 with no error.
counts = pd.concat([counts, pd.DataFrame([
    {"@type": "Inference (staging only)", "production": nde_staging.count("@type:Inference"),
     "what it is": "one differential-expression contrast"},
])], ignore_index=True)
counts

production index: 14,205,086 records across 48 repositories



,@type,production,what it is
0,Dataset,5442683,a study / series / accession
1,Sample,8729088,"one biological sample, e.g. a GEO GSM"
2,ComputationalTool,33272,software and workflows
3,ResourceCatalog,43,"a whole repository, described as a record"
4,Inference (staging only),10399895,one differential-expression contrast


In [3]:
top = nde.facet("@type:Dataset", "includedInDataCatalog.name", facet_size=8)
pd.DataFrame(top.items(), columns=["repository", "datasets"])

,repository,datasets
0,Figshare,2177036
1,NCBI BioProject,1108238
2,NCBI SRA,658408
3,Zenodo,649929
4,NCBI GEO,293799
5,Protein Data Bank,152503
6,Mendeley,151747
7,Omics Discovery Index (OmicsDI),129472


Two features of this index shape everything that follows.

**Samples outnumber datasets.** Most GEO content is indexed at GSM granularity as well as GSE, so
`@type:Sample` is the largest class. That is what makes Strategy 4's dataset discovery work at all: a
drug named on individual sample records is a much better signal of a treatment arm than the same
name appearing in a study abstract.

**Annotation coverage is partial**, which bounds how much can be done with structured fields alone:

In [4]:
n_ds = nde.count("@type:Dataset")
pd.DataFrame([
    {"field": f, "datasets with it": nde.count(f"@type:Dataset AND _exists_:{f}"),
     "% of datasets": round(100 * nde.count(f"@type:Dataset AND _exists_:{f}") / n_ds, 1)}
    for f in ["healthCondition", "species", "measurementTechnique"]
])

,field,datasets with it,% of datasets
0,healthCondition,844870,15.5
1,species,1996437,36.7
2,measurementTechnique,2321246,42.6


Only a minority of datasets carry an ontology-mapped disease, so discovery cannot rely on
`healthCondition` alone — free-text search has to be unioned in. And there is **no chemical or gene
field anywhere in the index**, which is the single most important constraint on this project: every
drug match below is free text, and that is where the expression strategies lose most of their
precision and recall. Strategy 1 escapes it only by leaving NDE entirely.

Where each strategy sits:

| strategy | NDE involvement |
|---|---|
| 1 · activity | **none** — PubChem BioAssay and ChEMBL are not indexed by NDE |
| 2 · expression | `@type:Inference` on **staging** — GXA contrasts, 10.4M records, not yet in production |
| 3 · inverted | `@type:Inference` again, queried gene-first |
| 4 · discovery | `@type:Dataset` + `@type:Sample` on production |

## 1. What Translator returned

A creative-mode `biolink:treats` query on asthma, answered by 13 reasoning agents. We keep only
2-hop paths of the form `ChemicalEntity → Gene → Disease`, since those are the ones that name a
mechanism we could look for data about.

In [5]:
drugs = {p["drug_name"] for p in paths}
genes = {p["gene_name"] for p in paths}
edges = {(p["drug"], p["gene"]) for p in paths}
print(f"{len(drugs)} distinct drugs · {len(genes)} distinct genes · {len(edges)} distinct drug→gene edges")
print("contributing agents:", dict(collections.Counter(p["agent"] for p in paths)))

174 distinct drugs · 76 distinct genes · 220 distinct drug→gene edges
contributing agents: {'ars-ars-agent': 368, 'ara-unsecret': 117, 'ara-arax': 311}


In [6]:
# Top answers by the ARS score, with the gene(s) each path runs through.
best = {}
for p in paths:
    best[p["drug_name"]] = max(best.get(p["drug_name"], 0), p.get("score") or 0)

top10 = pd.DataFrame([
    {"drug": d, "score": round(s, 3),
     "genes": ", ".join(sorted({p["gene_name"] for p in paths if p["drug_name"] == d}))}
    for d, s in sorted(best.items(), key=lambda kv: -kv[1])[:10]
])
top10

,drug,score,genes
0,Terbutaline,0.994,"ADRB1, ADRB2"
1,Prednisolone,0.992,"NR3C1, NR3C2"
2,Adrenal Cortex Hormones,0.992,TNF
3,Prednisone,0.986,NR3C1
4,Zafirlukast,0.973,"CYSLTR1, CYSLTR2, MAPK1"
5,beclomethasone,0.971,NR3C1
6,Zileuton,0.969,ALOX5
7,Roflumilast,0.958,"PDE4A, PDE4B, PDE4D"
8,Triamcinolone,0.955,"CD1A, MMP1, NR3C1"
9,Betamethasone,0.949,NR3C1


The answer is pharmacologically sensible: β2-agonists (terbutaline), inhaled and systemic
corticosteroids (prednisolone, prednisone, beclomethasone, betamethasone, triamcinolone), a
leukotriene receptor antagonist (zafirlukast), a 5-lipoxygenase inhibitor (zileuton) and a PDE4
inhibitor (roflumilast) — each routed through the gene you would expect.

That is the set we now try to ground in data.

In [7]:
gene_counts = collections.Counter(p["gene_name"] for p in paths).most_common(8)
pd.DataFrame(gene_counts, columns=["gene", "paths through it"])

,gene,paths through it
0,P2RX3,164
1,NR3C1,90
2,ADRB2,56
3,HRH1,55
4,PTGS2,40
5,TNF,31
6,PDE4A,26
7,IL4R,26


## 2. Strategy 1 — does activity data support the claim?

Translator's drug→gene edges mostly assert changes in **activity**: this drug inhibits, agonises or
antagonises that protein. So the most direct test is an assay that measures binding and inhibition
— PubChem BioAssay for screening and dose-response, ChEMBL for curated mechanism of action.

Neither source is indexed by NDE; its catalog has LINCS and ReframeDB but no ChEMBL, PubChem or
BindingDB. This strategy therefore reaches outside NDE, which is itself a coverage gap worth
reporting given that activity is the modality Translator's edges are about.

The join is exact: Translator emits `NCBIGene:154`, PubChem's `Target GeneID` column is `154`, and
Node Normalizer supplies the compound's PubChem CID and ChEMBL id. No text matching anywhere — the
free-text problem that limits Strategies 2–4 simply does not arise.

In [8]:
route_d = json.loads((ARS / "route_d.json").read_text())["results"]
v = collections.Counter(r["verdict"] for r in route_d)
measured = v["mechanism_agrees"] + v["mechanism_disagrees"] + v["binding_confirmed"] + v["measured_inactive"]
print(f"edges evaluated: {len(route_d)}")
for k, n in v.most_common():
    print(f"  {k:20s} {n}")
print(f"\nedges with a directly measured compound–target result: "
      f"{measured}/{len(route_d)} ({100*measured/len(route_d):.0f}%)")
print(f"  vs Strategy 2 on the same answer set: 1/123 (1%)")

edges evaluated: 220
  binding_confirmed    129
  not_tested           34
  mechanism_untyped    27
  no_compound_id       20
  no_activity_data     8
  measured_inactive    2

edges with a directly measured compound–target result: 131/220 (60%)
  vs Strategy 2 on the same answer set: 1/123 (1%)


In [9]:
# Activity evidence for Translator's top-scoring asthma drugs: how the interaction is
# typed (ChEMBL) and how strongly it was measured (PubChem BioAssay, plus ChEMBL's
# standardised pChEMBL). The assay type is chosen to match the mechanism -- EC50/AC50
# for an agonist, IC50/Ki for an inhibitor -- since a single "most potent" value across
# all types can hand back a counter-screen.
from translator_nde.activity import PubChemBioAssay, preferred_potency, pchembl_to_um

pubchem = PubChemBioAssay("../data/activity")

def fmt(um):
    if um is None:
        return "–"
    return f"{um*1000:.3g} nM" if um < 1 else f"{um:.3g} µM"

mech = [r for r in route_d if r["chembl_action_type"]]
rows = []
for r in sorted(mech, key=lambda r: -(r["max_phase"] or 0))[:10]:
    by_type = pubchem.potency_by_type(r["drug_cid"], r["gene"]) if r["drug_cid"] else {}
    pref = preferred_potency(by_type, r["chembl_action_type"])
    rows.append({
        "drug": r["drug_name"], "gene": r["gene_name"],
        "action": r["chembl_action_type"],
        "assay": pref[0] if pref else "–",
        "PubChem": fmt(pref[1] if pref else None),
        "ChEMBL (pChEMBL)": fmt(pchembl_to_um(r["pchembl_max"])),
        "active/inactive": f"{r['n_active']}/{r['n_inactive']}",
    })
pd.DataFrame(rows)

,drug,gene,action,assay,PubChem,ChEMBL (pChEMBL),active/inactive
0,Terbutaline,ADRB2,AGONIST,EC50,3.16 µM,2.51 µM,5/0
1,Prednisolone,NR3C1,AGONIST,EC50,31.6 nM,0.525 nM,14/0
2,Prednisone,NR3C1,AGONIST,AC50,268 nM,28.8 nM,8/7
3,Zafirlukast,CYSLTR1,ANTAGONIST,IC50,0.26 nM,0.257 nM,13/0
4,Zileuton,ALOX5,INHIBITOR,IC50,150 nM,151 nM,94/0
5,Roflumilast,PDE4B,INHIBITOR,–,–,0.151 nM,1/0
6,Roflumilast,PDE4A,INHIBITOR,IC50,0.21 nM,0.209 nM,8/0
7,Triamcinolone,NR3C1,AGONIST,AC50,9.1 nM,9.12 nM,8/1
8,Mepolizumab,IL5,INHIBITOR,–,–,–,0/0
9,Aspirin,PTGS2,INHIBITOR,IC50,2.4 µM,2.4 µM,10/2


**Translator's top-scoring asthma answers are confirmed, quantitatively and with the right
mechanism.** Terbutaline is an ADRB2 agonist, zafirlukast a CysLT1 antagonist, zileuton a
5-lipoxygenase inhibitor, roflumilast a PDE4 inhibitor — each approved, each with a measured
potency.

The two sources agree where both have data, which is a useful check on the join: zileuton/ALOX5
0.15 vs 0.151 µM, roflumilast/PDE4A 0.21 vs 0.209 nM, aspirin/PTGS2 2.4 vs 2.399 µM,
triamcinolone/NR3C1 9.1 vs 9.12 nM. They are independent records reached by different identifiers —
PubChem via the NCBI Gene id, ChEMBL via the UniProt accession — so the agreement says the compound
and target were resolved correctly on both sides.

The blanks are informative too. Mepolizumab and dupilumab are monoclonal antibodies: no PubChem
CID, no small-molecule assay, so the row is empty by construction rather than by absence of
evidence. PubChem also records **`Inactive`** outcomes, which expression data structurally cannot —
GXA stores only significant results, so a missing record is uninformative, whereas a recorded
inactive measurement is a real negative.

**Hold on to this drug list.** Strategy 2 asks the same question of expression data and returns
almost nothing for exactly these drugs.

## 3. Strategy 2 — does the drug change the gene's expression?

This was the project's original premise. NDE's staging index carries `@type:Inference`: one record
per Gene Expression Atlas differential-expression contrast, with a log2 fold change, an adjusted
p-value, and Biolink-typed direction and aspect qualifiers. Those line up field-for-field with a
Translator drug→gene edge, so the question can be stronger than *"does data exist?"* — it can be
*"does the measured direction agree with the asserted one?"*

Run **forward**: take Translator's edges and ask GXA about each one.

In [10]:
route_a = json.loads((ARS / "route_a.json").read_text())["results"]
verdicts = collections.Counter(r["verdict"] for r in route_a)

# 97 of the 220 edges name a research compound or synthetic peptide rather than a drug
# (IUPAC strings, "H-D-Phe-His-Leu-Leu-Arg-…"); free-text search cannot match those.
print(f"edges evaluated: {len(route_a)}")
for v, n in verdicts.most_common():
    print(f"  {v:24s} {n}")

edges evaluated: 123
  no_drug_data             116
  tested_not_significant   6
  agrees                   1


In [11]:
informative = [r for r in route_a if r["verdict"] != "no_drug_data"]
pd.DataFrame([
    {"drug": r["drug"], "gene": r["gene"], "verdict": r["verdict"],
     "contrasts": r["n_contrasts"], "median log2FC": r["median_log2fc"],
     "GXA experiment": ", ".join(r["experiments"])}
    for r in informative
])

,drug,gene,verdict,contrasts,median log2FC,GXA experiment
0,Prednisolone,NR3C1,tested_not_significant,0,NaN,
1,Prednisolone,NR3C2,tested_not_significant,0,NaN,
2,Prednisone,NR3C1,tested_not_significant,0,NaN,
3,Dupilumab,IL4R,tested_not_significant,0,NaN,
4,Sorafenib,ATG5,tested_not_significant,0,NaN,
5,Rifampicin,PPARGC1A,tested_not_significant,0,NaN,
6,Cyclic AMP,PPARGC1A,agrees,2,1.85,E-MTAB-2602


**Of 123 evaluable drug→gene edges, 7 returned any GXA drug-perturbation data, and exactly 1 had
expression data that agreed with the assertion.**

The `tested_not_significant` rows are informative in their own right: GXA *does* hold contrasts for
those drugs, measured genome-wide, and the gene never came up as differentially expressed — weak
evidence *against* the edge rather than absence of evidence. (`n_contrasts` above counts contrasts
for that drug–gene pair; the verdict rests on a separate count across all genes.) Four are clean
dose-controlled designs — `prednisolone 2 micromolar` in ALL cell lines, `dupilumab 150 milligram`
in skin lesions, `Sorafenib 5 micromolar` in HuH-7, `rifampicin 5 micromolar`.

⚠️ **`Prednisone → NR3C1` is a false positive**, so the honest count is 6 with data, not 7. All of
prednisone's human GXA contrasts come from one design —
`'systemic-onset juvenile idiopathic arthritis; Prednisone, NSAID, methotrexate' vs 'normal; none'`
— where the variable is *disease* and prednisone only describes what the patients were taking. The
arm rule misses it because the drug is a standalone factor absent from the reference arm; it cannot
distinguish a drug administered from a drug naming the cohort when the control is untreated healthy
subjects.

The single agreement is `Cyclic AMP → PPARGC1A`, and cyclic AMP is a second messenger rather than a
therapeutic.

Note which drugs are **absent entirely**. Asthma's first-line therapies have no human GXA contrasts
whatsoever — this is a selection gap in Expression Atlas, not a failure of the matching.

In [12]:
# Test-arm contrast counts measured directly against NDE staging, any species.
gxa_coverage = pd.DataFrame([
    ("budesonide", 0), ("formoterol", 0), ("fluticasone", 0), ("montelukast", 0),
    ("salbutamol / albuterol", 6), ("theophylline", 222), ("aspirin", 213),
    ("prednisolone", 481), ("dexamethasone", 57679),
], columns=["drug", "GXA contrasts (any species)"])
gxa_coverage["human"] = [0, 0, 0, 0, 6, 0, 0, 232, 23314]
gxa_coverage

,drug,GXA contrasts (any species),human
0,budesonide,0,0
1,formoterol,0,0
2,fluticasone,0,0
3,montelukast,0,0
4,salbutamol / albuterol,6,6
5,theophylline,222,0
6,aspirin,213,0
7,prednisolone,481,232
8,dexamethasone,57679,23314


Budesonide, formoterol, fluticasone and montelukast: **zero contrasts in any species**. Salbutamol's
six are incidental name matches. Theophylline's and aspirin's are rat toxicology. Dexamethasone is
the outlier that makes the atlas look better stocked than it is — it is a cell-culture workhorse.

So the failure is not in the matching. **Expression Atlas does not contain the drugs clinicians use
for asthma** — the same drugs Strategy 1 characterised completely. Two different reasons compound
here: the atlas lacks these compounds, and even where it has a compound, an agonist or a kinase
inhibitor need not change its target's transcript at all.

Strategies 3 and 4 respond to that in two different ways: keep the data and change the question, or
keep the question and change the data.

## 4. Strategy 3 — what other compounds move these genes?

Keep GXA, change the question. Translator's top asthma answers route through **NR3C1** (the
glucocorticoid receptor, for every corticosteroid) and **ADRB2** (for the β2-agonists). If moving
those genes is therapeutically useful, GXA can be asked gene-first which *other* compounds move
them — and every hit Translator did not propose is a repurposing hypothesis.

Querying gene-first plays to GXA's one structured axis: `observationAbout` carries a gene symbol
and an Ensembl id, so no text matching is needed on the gene side, which is where Strategies 2 and
4 leak most of their recall.

In [13]:
route_c = json.loads((ARS / "route_c.json").read_text())
spec = {c["compound"]: c for c in route_c["compounds"]}

def alternates(gene, n=5):
    rows = [r for r in route_c["contrasts"] if r["gene"] == gene]
    best = {}
    for r in rows:                       # keep each compound's largest effect
        k = r["compound"]
        if k not in best or abs(r["log2fc"] or 0) > abs(best[k]["log2fc"] or 0):
            best[k] = r
    out = [r for r in best.values() if spec[r["compound"]]["kind"] == "compound"]
    out.sort(key=lambda r: -(spec[r["compound"]]["specificity"] or 0))
    return pd.DataFrame([
        {"gene": gene, "compound": r["compound"], "direction": r["goal_direction"],
         "log2FC": r["log2fc"], "GXA experiment": r["experiment"],
         "genes this compound moves": spec[r["compound"]]["gxa_genes_moved"],
         "specificity": spec[r["compound"]]["specificity"]}
        for r in out[:n]])

pd.concat([alternates("NR3C1"), alternates("ADRB2")], ignore_index=True)

,gene,compound,direction,log2FC,GXA experiment,genes this compound moves,specificity
0,NR3C1,Digoxin,Upregulated,1.1,E-MTAB-5982,1000,0.0080
1,NR3C1,docetaxel,Upregulated,1.2,E-GEOD-28784,1000,0.0080
2,NR3C1,5-Aza,Upregulated,2.1,E-GEOD-41364,1000,0.0060
3,NR3C1,retinoic acid,Upregulated,1.2,E-MEXP-3577,1000,0.0060
4,NR3C1,paclitaxel,Upregulated,1.4,E-GEOD-28784,1000,0.0040
5,ADRB2,doxycycline,Downregulated,-5.1,E-GEOD-60548,1000,0.0400
6,ADRB2,valproic acid,Upregulated,3.9,E-MTAB-5984,1000,0.0380
7,ADRB2,trichostatin A,Upregulated,3.2,E-GEOD-37376,1000,0.0260
8,ADRB2,4-hydroxy-3-methyl-but-2-enyl pyrophosphate,Downregulated,-2.3,E-MEXP-1601,1000,0.0190
9,ADRB2,parthenolide,Downregulated,-1.6,E-GEOD-7538,381,0.0184


**Read this cautiously — the yield is poor, and the reason is instructive.**

Ranking by "how many of the disease's genes does this compound move" puts doxycycline, valproic
acid and trichostatin A on top of every list. Doxycycline is the Tet-on induction agent in a large
number of experiments, not a therapeutic; valproic acid and trichostatin A are HDAC inhibitors that
move thousands of genes. They score highly because they move *everything*.

The `specificity` column corrects for that — what fraction of a compound's total GXA activity lands
on this disease's genes — and once applied, nothing rises above ~4%. The compounds that survive are
still mostly promiscuous.

A second limit: 70 of asthma's 76 genes carry **no direction qualifier** from Translator, so for
most genes we cannot say which way is therapeutically useful and have to report both. Strategy 3 is
hypothesis-generating at best, and on this disease it generates little.

## 5. Strategy 4 — is there a dataset that profiles these drugs?

Keep the question, change the data. Drop the requirement that anything adjudicate a specific edge
and just ask: for a drug Translator proposes, has anyone run an expression experiment on it? This
is the most permissive of the four, and the one most likely to return something usable.

Ask it of the whole head of the answer list — the **top 20 drugs** — rather than a hand-picked one.
Two stages, because the cheap signal is not trustworthy alone:

1. **NDE sample-level search.** A drug named on individual `@type:Sample` records is a much better
   hint than one named on the `Dataset`, which may only be an abstract mention. Group the hits by
   parent series.
2. **GEO confirmation.** Sample mentions are still ambiguous — a study-wide descriptor ("asthma
   patients on ICS") names the drug on every sample while perturbing nothing. The authoritative
   signal is a per-sample field that names the drug *and* takes at least one other value: a
   treatment arm with something to compare against. Depositors put that in
   `characteristics_ch1`, the sample title, or the source name, so all three are checked.

In [14]:
top20 = json.loads(Path("../results/asthma_top20_datasets.json").read_text())["rows"]
pd.DataFrame([
    {"#": r["rank"], "drug": r["drug"], "score": r["score"],
     "NDE samples (human)": r["nde_samples_human"], "(any species)": r["nde_samples_any"],
     "best series": r["best_series"] or "—",
     "experiment type": r["experiment_type"],
     "n treated / control": (f"{r['n_arm']} / {r['n_control']}" if r["n_arm"] else "—"),
     "experimental system": r["system"] or "—",
     "arm vs control": (f"{r['arm']} vs {r['control']}"
                        if r["arm"] and r["control"] else
                        (f"{r['arm']} (no control level)" if r["arm"] else "—"))}
    for r in top20
])

,#,drug,score,NDE samples (human),(any species),best series,experiment type,n treated / control,experimental system,arm vs control
0,1,Terbutaline,0.994,14,67,GSE20297,array,1 / 1,"HaCaT, cell line from epidermal keratinocyte",HaCaT_TNF+IFNg+terbutaline_1 vs HaCaT_unstimulated_1
1,2,Prednisolone,0.992,2123,2453,GSE40165,array,117 / 117,"blood, Vietnamese dengue patients",treated with: high-dose (2mg/kg) of prednisolone vs trea...
2,3,Adrenal Cortex Hormones,0.992,0,0,—,—,—,—,—
3,4,Prednisone,0.986,2344,2580,GSE224705,array,70 / 260,"Whole Blood, SLE",prednisone_dosis: 5 vs prednisone_dosis: 0
4,5,Zafirlukast,0.973,1,8,—,—,—,—,—
5,6,beclomethasone,0.971,1,96,GSE57219,array,12 / 11,Oncorhynchus mykiss: liver,treatment group: beclomethasone diproprionate exposure v...
6,7,Zileuton,0.969,132,268,GSE175616,array,59 / 64,Brushing of nasal epithelium,treatment: Aspirin+Zileuton vs treatment: Placebo
7,8,Roflumilast,0.958,114,141,GSE126981,RNA-seq,4 / 4,BEAS-2B human airway epithelial cell line,treatment: Roflumilast N-oxide (RNO) vs treatment: Vehicle
8,9,Triamcinolone,0.955,185,198,GSE136034,other,31 / 40,Human cervical cancer cell line,treatment: 100 nM triamcinolone acetonide(TA) for 4 hour...
9,10,Betamethasone,0.949,80,447,GSE32473,array,1 / 1,skin biopsy,"patient 6, arm treated with betamethasone vs patient 6, ..."


**Sixteen of the top 20 have a confirmed perturbation series**, but the last three columns are
where the table earns its keep, because a confirmed treatment arm is not the same as a usable
experiment.

**`experiment type`** is GEO's own declaration, and two of the sixteen are not expression studies at
all. `GSE136034` (triamcinolone) reads `other`: it is a genuine triamcinolone-vs-DMSO perturbation
with 31 v 40 clean arms, which is why it passes arm confirmation, but its only data file is a
4C-seq chromatin-capture table — it cannot answer a differential-expression question. `GSE270608`
(flunisolide) is **ChIP-seq**. Counting arms without checking modality would have put both forward
as candidates.

**`n treated / control`** exposes the other failure mode. Betamethasone's best series is `1 / 1`;
roflumilast — the most apt drug-and-tissue pairing in the table, a PDE4 inhibitor in human airway
epithelium — is `4 / 4`, enough to interrogate one gene but not to support pathway enrichment.
Pemirolast is `4 / 114`, its four samples being one compound in a ~300-drug mouse organoid screen.

**`experimental system`** carries the third caveat: only four of the sixteen sit in airway tissue —
zileuton in nasal epithelial brushings, roflumilast in the BEAS-2B airway line, mepolizumab in
nasal lavage, dupilumab in nasal brushings. The rest treat with an asthma drug somewhere else
entirely: prednisolone in blood from *dengue patients*, prednisone in *lupus* whole blood, aspirin
in *purified platelets*, terbutaline in *HaCaT keratinocytes*, tranilast in *iPSCs*. Five rows are
non-human, marked with the organism, and appear only where no human series exists — pemirolast's
sole perturbation dataset anywhere is a mouse organoid screen.

So dataset discovery succeeds at the level of *drug* far more often than at the level of *drug, in a
relevant system, measured the right way, with enough samples to analyse*. Each of those qualifiers
removes rows, and none of them is visible from a hit count.

Two entries are not drugs at all: "Adrenal Cortex Hormones" and "P [Preparations]" are vocabulary
artifacts that reached the answer as `ChemicalEntity` nodes.

> **A note on counting.** GEO's web interface returns much larger numbers — 90 records for
> terbutaline against the single series here. Most of that gap is granularity and species: 74 of
> the 90 are individual `GSM` samples rather than datasets, 1 is a platform, and of the 15
> series-level records only 4 are human. Of those, two are colon-biopsy cohorts and one an asthma
> T-cell cohort where terbutaline is a patient's concomitant medication, not an experimental arm.
> One — `GSE20297` — is a genuine terbutaline experiment, and it is the row above.

### One case where two proposed drugs share a trial

Budesonide and formoterol rank lower in the answer (score 0.53, outside the top 20), but they are
the most interesting case in the set: both are drugs Translator proposed for asthma, they appear as
**arms of the same trial**, and the tissue is airway epithelium.

In [15]:
for drug in ["Budesonide", "Formoterol"]:
    rows = [p for p in paths if p["drug_name"] == drug]
    g = sorted({p["gene_name"] for p in rows})
    print(f"{drug:12s} {len(g):2d} gene edges: {', '.join(g)}")

Budesonide   10 gene edges: ANXA1, CRHR1, CYP1A2, CYP2C19, CYP2C9, CYP3A4, EDN1, NR1I2, NR3C2, PGR
Formoterol    2 gene edges: ADRB1, CYP2C19


Both arms belong to one trial, and NDE indexes it:

In [16]:
import sys
sys.path.insert(0, "../src")
from translator_nde.nde import NDEClient, PROD

nde = NDEClient(base_url=PROD)
GSE = "GSE162120"
print(f"{GSE} in NDE production:")
print(f"  Dataset records: {nde.count(f'identifier:\"{GSE}\"')}")
print(f"  Sample records : {nde.count(f'@type:Sample AND isBasisFor.identifier:\"{GSE}\"')}")
rec = next(iter(nde.scroll(f'identifier:\"{GSE}\"', fields="name,includedInDataCatalog.name", max_records=1)))
print(f"  title          : {rec['name']}")

GSE162120 in NDE production:


  Dataset records: 1


  Sample records : 118


  title          : The DISARM study: effects of inhaled corticosteroids on bronchial epithelial cell gene expression in COPD


In [17]:
import re
# NDE keeps the timepoint as a structured field but not the treatment arm, which survives only
# inside the free-text `sampleProcess` blob. A regex recovers it; the counts match GEO exactly.
arms, timepoints = collections.Counter(), collections.Counter()
for s in nde.scroll(f'@type:Sample AND isBasisFor.identifier:"{GSE}"',
                    fields="sampleProcess,temporalCoverage", max_records=200):
    m = re.search(r"\b(FOR/BUD|SAL/FLU|FOR)\b", s.get("sampleProcess") or "")
    arms[m.group(1) if m else "unlabelled"] += 1
    t = s.get("temporalCoverage") or {}
    timepoints[(t[0] if isinstance(t, list) else t).get("duration")] += 1

print("treatment arms:", dict(arms))
print("timepoints    :", dict(timepoints))

treatment arms: {'SAL/FLU': 40, 'FOR/BUD': 36, 'FOR': 41, 'unlabelled': 1}
timepoints    : {'pre-treatment': 62, 'post-treatment': 56}


**This is the bridge closing** — with one honest qualification, visible in the title printed above.

The DISARM trial randomised patients to formoterol (FOR), formoterol/**budesonide** (FOR/BUD) or
salmeterol/fluticasone (SAL/FLU), with bronchial brushings before and after 12 weeks and a counts
matrix deposited. Two of its three arms are drugs Translator proposed for asthma, and NDE indexes
the dataset and all 118 samples.

⚠️ **It is a COPD cohort, not an asthma cohort.** The drugs, the tissue and the delivery route are
the ones Translator's asthma answer names, and the two conditions overlap clinically, but this is
not the same indication. It is the right *drug* evidence in an adjacent disease — which is
typically what dataset discovery returns, and worth being explicit about rather than counting as a
clean hit.

It is also the only dataset of its kind we found: across the drug-treatment series re-analysed for
the other diseases in this project, none tested a drug Translator had proposed for that disease.

Two further limitations:

- the arm labels are not a structured NDE field — they had to be scraped from the free-text
  `sampleProcess` blob, so the join is fragile;
- all twelve budesonide/formoterol edges carry `direction: None`, so this dataset can support a
  coverage test, not a direction test.

### Reanalysis: GSE292059, mepolizumab vs placebo

Discovery is only worth doing if the data can then be used. Of the sixteen candidates,
`GSE292059` is the one that clears every filter at once: RNA-seq, nasal lavage, 443 v 522 samples,
placebo-controlled, raw counts deposited. Translator proposes mepolizumab for asthma through
**IL5** and **IL5RA**.

The trial is longitudinal — 965 samples over 270 donors, visits 01 to 14 plus unscheduled colds —
so the primary contrast is the single timepoint at end of treatment (**visit14**), which keeps the
drug effect from mixing with season and intercurrent illness. **Baseline (visit01) is analysed
identically as a negative control**: before treatment the two arms should not differ.

Two questions: was the Translator-nominated gene differentially expressed, and what do all the
differentially expressed genes amount to?

In [18]:
re_out = json.loads(Path("../results/gse292059_reanalysis.json").read_text())
rows = []
for visit, label in [("visit14", "end of treatment"), ("visit01", "baseline (control)")]:
    c = re_out["contrasts"][visit]
    for gene, r in c["translator_genes"].items():
        rows.append({"contrast": label, "n treated/control": f"{c['n_treated']}/{c['n_control']}",
                     "DE genes (FDR<0.05)": c["n_significant"],
                     "gene": gene,
                     "logFC": None if r is None else round(r["logFC"], 3),
                     "adj p": None if r is None else f"{r['adj_p']:.2g}",
                     "verdict": "—" if r is None else
                                ("**DE**" if r["adj_p"] < 0.05 else "not DE")})
pd.DataFrame(rows)

,contrast,n treated/control,DE genes (FDR<0.05),gene,logFC,adj p,verdict
0,end of treatment,73/79,482,IL5,-0.084,0.85,not DE
1,end of treatment,73/79,482,IL5RA,-1.257,0.0012,**DE**
2,baseline (control),123/122,0,IL5,-0.027,0.97,not DE
3,baseline (control),123/122,0,IL5RA,-0.390,0.87,not DE


**One of the two nominated genes moves, and which one moves is the whole point.**

`IL5RA` falls sharply — logFC −1.26, adj p 0.0012. `IL5` does not budge (logFC −0.08, p 0.64).
Mepolizumab is an antibody that neutralises IL-5 *protein*; there is no reason for the cytokine's
transcript to change, and it doesn't. IL5RA is the receptor carried by eosinophils, and depleting
eosinophils removes the cells that express it — so the receptor transcript drops even though the
drug never touches its transcription.

This is the project's central distinction, measured: an activity-level intervention is invisible at
its own target's transcript and plainly visible one step away. A pipeline that had scored this edge
by asking "is IL5 differentially expressed?" would have called it a failure.

**The baseline contrast returns 0 differentially expressed genes** out of 13,013 tested, with 123 v
122 samples — more samples than the primary contrast. Randomisation worked, the pipeline is not
manufacturing signal, and the visit14 result is a treatment effect rather than a cohort difference.

In [19]:
c = re_out["contrasts"]["visit14"]
print(f"visit14: {c['n_significant']} DE genes — {c['n_up']} up, {c['n_down']} down in mepolizumab\n")
pd.DataFrame(c["top_de"][:12]).assign(logFC=lambda d: d.logFC.round(2))

visit14: 482 DE genes — 17 up, 465 down in mepolizumab



,gene,logFC,adj_p
0,RNASE2,-1.88,0.000277
1,TM6SF1,-0.81,0.000277
2,PRSS33,-1.87,0.000277
3,LGALS12,-1.92,0.000277
4,CLC,-2.10,0.000277
5,ADORA3,-1.57,0.000277
6,ADGRE1,-1.16,0.000277
7,VSTM1,-1.64,0.000277
8,CEBPE,-1.72,0.000277
9,SIGLEC8,-1.75,0.000357


**465 of the 482 are down.** Depleting a cell type produces a one-directional signature, and the
top genes are unambiguous about which cell: `RNASE2` (eosinophil-derived neurotoxin), `CLC`
(Charcot-Leyden crystal protein / galectin-10), `SIGLEC8`, `PRSS33`, `LGALS12`, `CEBPE`, `ADGRE1`,
`ADORA3`, `VSTM1` — eosinophil granule and lineage genes, all falling. That is the expected
pharmacology of an anti-IL-5 antibody, recovered from public data without any prior specification
of what to look for.

In [20]:
enr = c["enrichment"]["down:Reactome_2022"]
pd.DataFrame([{"pathway": t["term"], "genes": t["n_genes"],
               "adj p": f"{t['adj_p']:.2e}"} for t in enr[:8]])

,pathway,genes,adj p
0,Signaling By GPCR R-HSA-372790,37,1.85e-03
1,Signal Transduction R-HSA-162582,90,3.26e-03
2,Immune System R-HSA-168256,74,4.01e-03
3,Neutrophil Degranulation R-HSA-6798695,26,8.70e-03
4,GPCR Downstream Signaling R-HSA-388396,31,9.41e-03
5,Innate Immune System R-HSA-168249,44,1.19e-02
6,Drug-mediated Inhibition Of CDK4/CDK6 Activity R-HSA-975...,3,1.43e-02
7,Immunoregulatory Interactions Between A Lymphoid And A n...,11,1.47e-02


Enrichment on the 465 down-regulated genes gives **12 Reactome terms at adj p < 0.05**, headed by
*Signaling by GPCR*, *Immune System*, *Neutrophil Degranulation* and *Innate Immune System*.

Two honest caveats. *Neutrophil Degranulation* is the label Reactome puts on a granulocyte-granule
gene set that eosinophils largely share, so it understates the specificity the gene-level view
makes obvious — the pathway annotation is coarser than the data. And **GO Biological Process
returns nothing at adj p < 0.05** on the same gene list; the signature is real but sits in
cell-identity genes that GO's process ontology does not cluster tightly. Reporting only the library
that worked would misrepresent how robust the enrichment is.

So the answer to the second question is: the differentially expressed genes are an eosinophil
depletion signature, clearest at the level of individual marker genes, visible in Reactome, and
invisible to GO BP.

## Summary

| | strategy | asks | asthma result |
|---|---|---|---|
| **1** | activity | does activity data support the claim? | **131 / 220 edges (60%)** measured; 27 curated mechanisms with potencies, recovering Translator's top answers exactly |
| **2** | expression | does the drug change the gene's expression? | **1 / 123 edges** agreed; 6 had any data. Asthma's first-line drugs are absent from GXA entirely |
| **3** | inverted | what other compounds move these genes? | 73 compounds surfaced, but specificity ≤4% — dominated by HDAC inhibitors and the Tet-on inducer |
| **4** | discovery | is there a dataset profiling this drug? | 16 of the top 20 have a perturbation series, but only 4 in airway tissue and 2 are not expression studies. **GSE292059 reanalysed**: IL5RA down (adj p 0.001), IL5 unchanged, 465 genes down in an eosinophil-depletion signature |

**What the exercise shows.** The bridge is real, but it does not run where the original sketch
assumed. That sketch was Strategy 2: take an asserted edge, find the expression contrast that tests
it. It grounds 1 of 123 asthma edges — partly because Expression Atlas does not stock inhaled
asthma therapeutics, and partly because the edges assert *activity* while the atlas measures
*abundance*.

Strategy 1 answers the question the edges actually ask, at **fifty times the coverage** and with
quantitative potencies — but sits entirely outside NDE, which is the most actionable finding here
for the NDE team. Strategy 3 salvages the expression data by inverting the query, and on this
disease yields little that survives a promiscuity correction. Strategy 4 abandons edge-level
adjudication and does find real data.

The one place all three pieces line up — a Translator-proposed drug, a public dataset that perturbs
with it, and NDE indexing that dataset — is GSE162120, and even there the cohort is COPD rather than
asthma. Making that case routine rather than exceptional is the work.